In [ ]:
#packages
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.functions import col
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

In [ ]:
#read in income data
result_df=spark.read.csv("../data/curated/merged_postcode_income.csv", header=True)

In [ ]:
#read in transaction data by user and business
agg_data=spark.read.parquet('../data/curated/agg_by_userbiz/')
agg_data

In [ ]:
#aggregate by user id
user_agg=agg_data.groupBy('user_id').agg(
    f.sum('count').alias('count'),
    f.avg('mean').alias('mean')
)
user_agg

In [ ]:
user_agg_pd=user_agg.toPandas()

In [ ]:
#plot a graph of ave transaction amount vs. no. of transactions
sns.scatterplot(data=user_agg_pd, x='mean', y='count')
plt.xlabel('Average Transaction Amount')
plt.ylabel('Number of Transactions')
plt.title('User Information')
plt.show()

In [ ]:
biz_agg=agg_data.groupBy('merchant_abn').agg(
    f.sum('count').alias('count'),
    f.avg('mean').alias('mean')
)
biz_agg

In [ ]:
biz_agg_pd=biz_agg.toPandas()

In [ ]:
sns.scatterplot(data=biz_agg_pd, x='mean', y='count')
plt.xlabel('Average Transaction Amount')
plt.ylabel('Number of Transactions')
plt.title('Business Information')
plt.show()

In [ ]:
pdf = result_df.toPandas()
# Drop missing values (important to avoid errors)
pdf = pdf.dropna(subset=["median_total_income_2020"])

# Histogram
plt.figure(figsize=(8, 6))
plt.hist(pdf["median_total_income_2020"], bins=20, edgecolor="black", alpha=0.7)

plt.xlabel("Median Total Income (2020)")
plt.ylabel("Number of Postcodes")
plt.title("Distribution of Median Total Income by Postcode (2020)")
plt.tight_layout()
plt.show()

In [ ]:
# Drop missing values
pdf = pdf.dropna(subset=["median_total_income_2020"])

# Define bin edges in 10k intervals
min_val = float(pdf["median_total_income_2020"].min())
max_val = float(pdf["median_total_income_2020"].max())

# Start from the nearest 10k below min, go to nearest 10k above max
bins = np.arange((min_val // 10000) * 10000, ((max_val // 10000) + 1) * 10000 + 1, 10000)

# Create labels like "30k-40k", "40k-50k", etc.
labels = [f"{int(bins[i]/1000)}k-{int(bins[i+1]/1000)}k" for i in range(len(bins)-1)]

# Categorize incomes into bins
pdf["income_bin"] = pd.cut(pdf["median_total_income_2020"], bins=bins, labels=labels, right=False)

# Count how many postcodes fall into each bin
bin_counts = pdf["income_bin"].value_counts().sort_index()

# Bar chart
plt.figure(figsize=(10, 6))
bin_counts.plot(kind="bar", color="skyblue", edgecolor="black")

plt.xlabel("Income Bin")
plt.ylabel("Number of Postcodes")
plt.title("Postcodes Grouped into 10k Income Bins (2020)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
